# Práctica 6 - Preprocesamiento de datos para Machine Learning
**Sistemas de Aprendizaje Automático · Sesión 6**   
**Dataset:** `clientes_ecommerce.csv` · **Modalidad:** individual

## Contexto
Trabajas en una empresa de comercio electrónico que quiere predecir si un cliente comprará el próximo mes. El dataset ya se exploró en las sesiones 4 y 5. Ahora debes transformarlo para que esté listo para entrenar un modelo de clasificación binaria.

**No entrenes modelos en esta práctica.** Completa los bloques `TODO` y responde las preguntas en las celdas Markdown de respuesta.

### Objetivos
- Detectar ausentes y anomalías
- Elegir estrategias de imputación 
- Codificar variables nominales y ordinales
- Comparar escalados
- Separar entrenamiento/prueba
- Construir un `Pipeline` con `ColumnTransformer` 
- Evitar data leakage
- Verificar las matrices resultantes

> Coloca `clientes_ecommerce.csv` en la misma carpeta que este notebook. Usa scikit-learn 1.2 o posterior para `sparse_output=False`.

## Parte 1 - Diagnóstico del dataset
### Ejercicio 1. Carga y estructura
Carga el CSV, muestra 10 filas, dimensiones, tipos de datos, información general y número de registros duplicados.

In [53]:
%pip install scikit-learn


Note: you may need to restart the kernel to use updated packages.


In [54]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# TODO: cargar clientes_ecommerce.csv en df
df = pd.read_csv("clientes_ecommerce.csv")
# TODO: mostrar 10 primeras filas
df.head(10)
# TODO: consultar shape, dtypes, info y número de duplicados
print(df.shape)
print(df.dtypes)
print(df.info())
print("Filas duplicadas: ", df.duplicated().sum())

(400, 12)
cliente_id              int64
edad                    int64
antiguedad_meses      float64
ingresos_anuales      float64
gasto_mensual         float64
num_compras             int64
devoluciones            int64
dispositivo               str
zona                      str
nivel_fidelidad           str
satisfaccion            int64
compra_proximo_mes        str
dtype: object
<class 'pandas.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   cliente_id          400 non-null    int64  
 1   edad                400 non-null    int64  
 2   antiguedad_meses    395 non-null    float64
 3   ingresos_anuales    396 non-null    float64
 4   gasto_mensual       397 non-null    float64
 5   num_compras         400 non-null    int64  
 6   devoluciones        400 non-null    int64  
 7   dispositivo         400 non-null    str    
 8   zona                400 non

**Responde:** ¿Cuántas muestras y columnas hay? ¿Cuál es el target? ¿Qué columna identifica a cada cliente? ¿Hay filas duplicadas?

**Respuesta:**

- El dataset tiene **400 muestras** (filas) y **12 columnas**.
- El target es **`compra_proximo_mes`**, con valores `No` / `Sí` (clasificación binaria).
- La columna que identifica a cada cliente es **`cliente_id`** (no tiene valores repetidos).
- **No hay filas duplicadas** (`df.duplicated().sum()` devuelve 0).

### Ejercicio 2. Valores ausentes
Calcula el número de ausentes por columna, el porcentaje y el número de filas con al menos un ausente. Completa la tabla.

In [55]:
# TODO: ausentes por columna
ausentes = df.isna().sum()
# TODO: porcentaje de ausentes por columna
porcentaje = df.isna().mean().mul(100)
# TODO: filas con algún valor ausente
filas_incompletas = df.isna().any(axis=1).sum()

display(pd.DataFrame({"ausentes": ausentes, "porcentaje": porcentaje.round(2)}))
print("Filas con al menos un ausente:", filas_incompletas)

,ausentes,porcentaje
cliente_id,0,0.00
edad,0,0.00
antiguedad_meses,5,1.25
ingresos_anuales,4,1.00
gasto_mensual,3,0.75
num_compras,0,0.00
devoluciones,0,0.00
dispositivo,0,0.00
zona,0,0.00
nivel_fidelidad,0,0.00


Filas con al menos un ausente: 12


| Variable | Nº ausentes | Porcentaje |
|---|---:|---:|
| `antiguedad_meses` | 5 | 1,25 % |
| `ingresos_anuales` | 4 | 1,00 % |
| `gasto_mensual` | 3 | 0,75 % |

**¿Eliminarías todas las filas incompletas? ¿Por qué?**

Respuesta: No. Hay 12 filas con algún ausente (un 3 % del dataset) y en cada una falta solo un valor; el resto de columnas son válidas. Eliminarlas supondría perder información útil sin necesidad, y además podría sesgar la muestra si los ausentes no son aleatorios. Como el porcentaje de ausentes por columna es muy bajo, es mejor imputarlos (por ejemplo, con la mediana calculada solo sobre entrenamiento).

### Ejercicio 3. Edad sospechosa
Localiza edades superiores a 120 años, muestra `cliente_id` y `edad` y consulta mínimo y máximo.

In [56]:
# TODO: mínimo y máximo de edad
print("Edad mínima:", df["edad"].min())
print("Edad máxima:", df["edad"].max())
# TODO: localizar registros con edad > 120
df.loc[df["edad"] > 120, ["cliente_id", "edad"]]

Edad mínima: 18
Edad máxima: 195


,cliente_id,edad
137,20138,195


**Responde:** ¿Qué harías con esa edad? ¿Por qué no deberías cambiar 195 por 95 sin evidencia?

Respuesta:

- Una edad de 195 años es imposible, así que la trataría como un **error de dato**: la marcaría como valor ausente (`NaN`) y después la imputaría igual que el resto de ausentes (por ejemplo, con la mediana del conjunto de entrenamiento). No eliminaría la fila entera, porque el resto de sus datos sí son válidos.
- No debería cambiarla por 95 porque eso es **inventarse el dato**: suponer que sobra un "1" es solo una hipótesis. El valor real podría ser 19, 59 o cualquier otro. Sin una fuente que lo confirme (el formulario original, el sistema de registro...), corregirlo "a ojo" mete información falsa en el dataset como si fuera cierta.

## Parte 2 - Variables y división
### Ejercicio 4. Clasificación de columnas
Clasifica las columnas según su naturaleza.

| Tipo | Columnas |
|---|---|
| Numéricas | `edad`, `antiguedad_meses`, `ingresos_anuales`, `gasto_mensual`, `num_compras`, `devoluciones`, `satisfaccion` |
| Categóricas nominales | `dispositivo`, `zona` |
| Categóricas ordinales | `nivel_fidelidad` |
| Identificador | `cliente_id` |
| Target | `compra_proximo_mes` |

**Responde:** ¿Por qué `zona` es nominal y `nivel_fidelidad` ordinal? ¿Por qué no incluirías `cliente_id` en el modelo?

Respuesta:

- `zona` (Norte, Sur, Este, Oeste, Centro) es **nominal** porque sus categorías no tienen ningún orden natural: ninguna zona es "mayor" o "mejor" que otra.
- `nivel_fidelidad` es **ordinal** porque sus categorías sí siguen un orden lógico: Bronce < Plata < Oro < Platino.
- `cliente_id` es solo un identificador único asignado a cada cliente; su valor no tiene relación con el comportamiento de compra. Si se incluyera, el modelo podría aprender patrones falsos (o memorizar clientes concretos) y no generalizaría a clientes nuevos.

*Nota:* `satisfaccion` (1-5) es conceptualmente ordinal, pero en esta práctica se tratará como numérica.

### Ejercicio 5. Separar X e y
Excluye de X el identificador y el target. Codifica `No` como 0 y `Sí` como 1. Comprueba dimensiones y distribución de y.

In [57]:
# TODO: X = ... (sin cliente_id ni compra_proximo_mes)
X = df.drop(columns=["cliente_id", "compra_proximo_mes"])
# TODO: y = ... (mapear No:0, Sí:1)
y = df["compra_proximo_mes"].map({"No": 0, "Sí": 1})
# TODO: X.shape, y.shape, y.value_counts()
print("X:", X.shape)
print("y:", y.shape)
print(y.value_counts())

X: (400, 10)
y: (400,)
compra_proximo_mes
0    246
1    154
Name: count, dtype: int64


**¿Cuántas características predictoras iniciales hay?**

Respuesta: Hay **10 características predictoras** (las 12 columnas menos `cliente_id` y `compra_proximo_mes`). `X` tiene forma (400, 10) e `y` tiene 400 etiquetas: 246 clientes con `No` (0) y 154 con `Sí` (1), es decir, un 61,5 % / 38,5 %, así que las clases están algo desbalanceadas.

### Ejercicio 6. División entrenamiento/prueba
Separa 80 % para entrenamiento y 20 % para prueba. Utiliza `random_state=42` y `stratify=y`. Comprueba tamaños.

In [58]:
# TODO: completar
X_train, X_test, y_train, y_test = train_test_split(
     X, y, test_size=0.2, random_state=42, stratify=y
)
# TODO: mostrar las dimensiones de los cuatro conjuntos

print("Entrenamiento :" ,X_train.shape, y_train.shape)
print("test  :" ,X_test.shape, y_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

Entrenamiento : (320, 10) (320,)
test  : (80, 10) (80,)
compra_proximo_mes
0    0.615625
1    0.384375
Name: proportion, dtype: float64
compra_proximo_mes
0    0.6125
1    0.3875
Name: proportion, dtype: float64


**Responde:** ¿Cuántas muestras hay en cada conjunto? ¿Para qué sirve `stratify=y`? ¿Por qué dividimos antes de ajustar transformaciones?

Respuesta: ...

## Parte 3 - Imputación y encoding
### Ejercicio 7. Validación de edad
Sobre **copias** de `X_train` y `X_test`, sustituye edades superiores a 120 por `np.nan`. No imputes todavía: eso se hará en el pipeline. El umbral es una regla didáctica de validación del dominio.

In [ ]:
X_train = X_train.copy()
X_test = X_test.copy()

# TODO: marcar edades > 120 como NaN en ambos conjuntos
print(X_train.loc[df["edad"]>120,["edad"]])
X_train.loc[df["edad"]>120,["edad"]] = np.nan
print(X_train["edad"].isna().sum())

print(X_test.loc[df["edad"]>120,["edad"]])
X_test.loc[df["edad"]>120,["edad"]]=np.nan
print(X_test["edad"].isna().sum())

# TODO: contar ausentes en edad en entrenamiento y prueba

     edad
137   195
1
Empty DataFrame
Columns: [edad]
Index: []
0


### Ejercicio 8. Estrategias de imputación
Utilizaremos mediana para numéricas y moda para categóricas. Crea ambos objetos `SimpleImputer` **sin ajustarlos todavía**.

In [60]:
# TODO: imputer_numerico = SimpleImputer(...)
imputer_numerico = SimpleImputer(strategy="median")

# TODO: imputer_categorico = SimpleImputer(...)
imputer_categorico = SimpleImputer(strategy="most_frequent")

| Variable | Estrategia | Justificación |
|---|---|---|
| edad | ... | ... |
| antiguedad_meses | ... | ... |
| ingresos_anuales | ... | ... |
| gasto_mensual | ... | ... |
| dispositivo | ... | ... |
| zona | ... | ... |

**¿Por qué no ajustar el imputador con entrenamiento y prueba a la vez?**

Respuesta: ...

### Ejercicio 9. One-Hot Encoding
Prueba `OneHotEncoder` con un DataFrame independiente. Configura `handle_unknown="ignore"` y `sparse_output=False`. Muestra matriz y nombres de columnas.

In [61]:
ejemplo = pd.DataFrame({
    "dispositivo": ["Móvil", "Ordenador", "Tablet", "Móvil"]
})

# TODO: encoder_nominal = OneHotEncoder(...)
encoder_nominal = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
# TODO: aplicar fit_transform a ejemplo
matriz_ejemplo = encoder_nominal.fit_transform(ejemplo)
# TODO: consultar get_feature_names_out()
print(encoder_nominal.get_feature_names_out())
pd.DataFrame(matriz_ejemplo, columns=encoder_nominal.get_feature_names_out())

['dispositivo_Móvil' 'dispositivo_Ordenador' 'dispositivo_Tablet']


,dispositivo_Móvil,dispositivo_Ordenador,dispositivo_Tablet
0,1.0,0.0,0.0
1,0.0,1.0,0.0
2,0.0,0.0,1.0
3,1.0,0.0,0.0


**Responde:** ¿Cuántas columnas se generan? ¿Qué representan? ¿Por qué no codificar Móvil=0, Ordenador=1 y Tablet=2 como categorías ordenadas?

Respuesta: ...

### Ejercicio 10. Codificación ordinal
Configura `OrdinalEncoder` para el orden Bronce, Plata, Oro, Platino. Comprueba su funcionamiento con un ejemplo independiente.

In [62]:
ejemplo_fidelidad = pd.DataFrame({
    "nivel_fidelidad": ["Oro", "Bronce", "Platino", "Plata"]
})

# TODO: encoder_ordinal = OrdinalEncoder(categories=[[...]])
encoder_ordinal = OrdinalEncoder(categories=[["Bronce", "Plata", "Oro", "Platino"]],
        handle_unknown="use_encoded_value",
        unknown_value=-1)
# TODO: transformar ejemplo_fidelidad y mostrar el resultado
resultado = encoder_ordinal.fit_transform(ejemplo_fidelidad)
print(resultado)
print(resultado.ravel())

[[2.]
 [0.]
 [3.]
 [1.]]
[2. 0. 3. 1.]


**Responde:** ¿Qué representan los códigos? ¿La distancia entre niveles consecutivos es necesariamente igual?

Respuesta: ...

## Parte 4 - Escalado
### Ejercicio 11. Min-Max frente a Z-score
Aplica ambos escaladores al siguiente ejemplo. Redondea a tres decimales y completa la tabla.

In [63]:
datos = pd.DataFrame({"valor": [10, 20, 30, 40, 50]})

# TODO: crear MinMaxScaler y StandardScaler
minmax = MinMaxScaler()
zscore = StandardScaler()
# TODO: transformar datos con cada uno
comparacion = {
    "Original": datos["valor"],
    "Min-Max": minmax.fit_transform(datos).ravel(),
    "StandardScaler": zscore.fit_transform(datos).ravel()   
}
# TODO: mostrar tabla comparativa
tabla = pd.DataFrame(comparacion).round(3)
tabla

,Original,Min-Max,StandardScaler
0,10,0.00,-1.414
1,20,0.25,-0.707
2,30,0.50,0.000
3,40,0.75,0.707
4,50,1.00,1.414


| Original | Min-Max | StandardScaler |
|---:|---:|---:|
| 10 | 0.000 | ... |
| 20 | 0.250 | ... |
| 30 | 0.500 | ... |
| 40 | 0.750 | ... |
| 50 | 1.000 | ... |

**Responde:** ¿Qué rango da Min-Max en este ejemplo? ¿Qué valor toma la media tras estandarizar? ¿Pueden aparecer valores negativos? ¿Qué escalado elegirías para un algoritmo basado en distancias y por qué?

Respuesta: ...

### Ejercicio 12. Variables que escalaremos
Para esta práctica trataremos `satisfaccion` como numérica, aunque conceptualmente es una escala ordinal de 1 a 5. Es una simplificación que deberá revisarse en otros modelos.

In [64]:
numericas = [
    "edad", "antiguedad_meses", "ingresos_anuales",
    "gasto_mensual", "num_compras", "devoluciones", "satisfaccion"
]
nominales = ["dispositivo", "zona"]
ordinales = ["nivel_fidelidad"]

# TODO: comprobar que las columnas existen en X_train

**Responde:** ¿Qué variables tienen escalas más distintas? ¿Es obligatorio escalar para cualquier algoritmo? ¿Qué particularidad tiene `satisfaccion`?

Respuesta: ...

## Parte 5 - Pipeline completo
### Ejercicio 13. Pipeline numérico
Construye un `Pipeline` que impute con mediana y después estandarice.

In [ ]:
pipeline_numerico = Pipeline(steps=[
     ("imputer", imputer_numerico),
     ("scaler", zscore)
 ])

**¿Por qué imputamos antes de escalar en este pipeline?**

Respuesta: ...

### Ejercicio 14. Pipeline nominal
Imputa con moda y después aplica One-Hot Encoding (`handle_unknown="ignore"`, `sparse_output=False`).

In [66]:
pipeline_nominal = Pipeline(steps=[
     ("imputer", imputer_categorico),
     ("encoder", encoder_nominal)
 ])

### Ejercicio 15. Pipeline ordinal
Imputa con moda y codifica Bronce → Plata → Oro → Platino. Gestiona categorías desconocidas con `handle_unknown="use_encoded_value"` y `unknown_value=-1`.

In [67]:
pipeline_ordinal = Pipeline(steps=[
     ("imputer", imputer_categorico),
     ("encoder", encoder_ordinal)
 ])

### Ejercicio 16. ColumnTransformer
Combina los tres pipelines para aplicar cada transformación únicamente a sus columnas.

In [68]:
preprocesador = ColumnTransformer(transformers=[
     ("num", pipeline_numerico, numericas),
     ("nom", pipeline_nominal, nominales),
     ("ord", pipeline_ordinal, ordinales)
 ])

### Ejercicio 17. Transformar entrenamiento y prueba
Ajusta el preprocesador **solo** con entrenamiento y aplícalo a prueba sin reajustarlo.

In [69]:
X_train_preparado = preprocesador.fit_transform(X_train)
print(X_train_preparado[0])
X_test_preparado = preprocesador.transform(X_test)

[0.47368421 0.1779661  0.4978199  0.08323172 0.46666667 0.2
 0.5        0.         1.         0.         0.         0.
 1.         0.         0.         3.        ]


**Responde:** ¿Por qué no usar `fit_transform()` en prueba? ¿Qué aprende `fit()`? ¿Qué ocurre si aparece una categoría nominal nueva en prueba?

Respuesta: ...

## Parte 6 - Validación e informe
### Ejercicio 18. Verificar matrices
Comprueba tamaños, ausentes, tipos numéricos, correspondencia con las etiquetas y nombres de características.

In [70]:
# TODO: mostrar X_train_preparado.shape y X_test_preparado.shape
# TODO: contar np.isnan(...) en ambas matrices
# TODO: consultar dtype y get_feature_names_out()
# TODO: verificar filas de matrices frente a len(y_train), len(y_test)
# TODO: verificar que entrenamiento y prueba tienen igual número de columnas

**Responde:** ¿Por qué aumentaron las columnas? ¿Coinciden las características en ambos conjuntos? ¿Quedan ausentes? ¿Están alineadas las etiquetas?

Respuesta: ...

### Ejercicio 19. Reconstruir DataFrames
Convierte las matrices preparadas en `df_train_preparado` y `df_test_preparado`. Usa `preprocesador.get_feature_names_out()` y conserva los índices de X_train/X_test.

In [71]:
# TODO: columnas_finales = preprocesador.get_feature_names_out()
# TODO: df_train_preparado = pd.DataFrame(..., columns=..., index=X_train.index)
# TODO: df_test_preparado = pd.DataFrame(..., columns=..., index=X_test.index)
# TODO: mostrar head() de ambos